# UW-LYT-MS V2 on EUVP
Kaggle runner for the RGB-only, LYT-inspired multiscale UW-LYT V2 model. Enable Internet and a GPU accelerator. The full run uses exactly the same training configuration as the previous UW-LYT-MS experiment; only the model architecture changes.

In [ ]:
!pip install thop -q

In [ ]:
from pathlib import Path
import os
import subprocess

REPO = Path('/kaggle/working/underwater-image-enhancement')
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'refactored-paper-core',
        '--single-branch', '--depth', '1',
        'https://github.com/heniath/underwater-image-enhancement.git', str(REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
%pip install -q -e .

commit = subprocess.run(
    ['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip()
print('Repository commit:', commit)
assert 'uwlytmsv2_3ch' in subprocess.run(
    ['uwir-profile', '--list'], check=True, capture_output=True, text=True
).stdout, 'The checked-out branch does not contain UW-LYT-MS V2'

In [ ]:
import torch

candidates = [
    Path('/kaggle/input/euvp-dataset/EUVP'),
    Path('/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP'),
]
EUVP_ROOT = next((p for p in candidates if p.exists()), None)
if EUVP_ROOT is None:
    matches = list(Path('/kaggle/input').glob('**/EUVP/test_samples'))
    EUVP_ROOT = matches[0].parent if matches else None
assert EUVP_ROOT and (EUVP_ROOT / 'test_samples/Inp').exists(), 'EUVP dataset not found'

NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS > 0, 'Enable a Kaggle GPU accelerator'
SMOKE = False  # Set True for an optional one-epoch preflight.
EPOCHS, RUNS, SEEDS = (1, 1, '0') if SMOKE else (50, 1, '0')
TAG = 'smoke' if SMOKE else 'full'
CHECKPOINTS = Path(f'/kaggle/working/checkpoints_uwlytmsv2_{TAG}')
RESULTS = Path(f'/kaggle/working/results_uwlytmsv2_{TAG}')
print('Dataset:', EUVP_ROOT)
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)])
print('Mode:', TAG)

In [ ]:
!python -m scripts.experiments.ablation_euvp \
    --data_train_euvp "{EUVP_ROOT}" \
    --checkpoint_dir "{CHECKPOINTS}" \
    --val_folder "{RESULTS}" \
    --variants uwlytmsv2_3ch \
    --nEpochs {EPOCHS} --batchSize 16 --cropSize 256 \
    --lr 1e-4 --weight_decay 1e-5 \
    --L1_weight 1.0 --perceptual_weight 1.0 --SSIM_weight 0.0 \
    --scheduler_step 30 --scheduler_gamma 0.5 \
    --early_stop_patience 20 --num_runs {RUNS} --seeds {SEEDS} \
    --threads 2 --num_gpus {NUM_GPUS}

In [ ]:
!uwir-evaluate \
    --eval_benchmark euvp --data_train_euvp "{EUVP_ROOT}" \
    --checkpoint_dir "{CHECKPOINTS}" \
    --val_folder "{RESULTS / 'evaluation'}" \
    --batchSize 16 --cropSize 256 --threads 2 \
    --gpu_mode True --native_eval True

!uwir-profile uwlytmsv2 --device cuda --no-pretrained \
    --img-size 256 --runs 100 --output-dir "{RESULTS / 'profile'}"